In [ ]:
!pip install -q pandas numpy matplotlib seaborn nltk textblob vaderSentiment \
    scikit-learn torch transformers gradio bitsandbytes accelerate \
    gensim sentencepiece

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import pandas as pd

df = pd.read_csv('amazon.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('amazon.csv')

print("Initial shape:", df.shape)
print("Columns:", df.columns.tolist())

df['discounted_price'] = df['discounted_price'].str.replace(r'[₹,]', '', regex=True).astype(float)
df['actual_price'] = df['actual_price'].str.replace(r'[₹,]', '', regex=True).astype(float)
df['discount_percentage'] = df['discount_percentage'].str.replace('%', '').astype(float)
df['rating_count'] = df['rating_count'].str.replace(',', '').astype(float)

df['rating'] = df['rating'].replace('|', np.nan)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

initial_rows = len(df)
df = df.dropna(subset=['rating', 'review_content'])
print(f"\nDropped {initial_rows - len(df)} rows missing rating or reviews")
print("Shape after critical drop:", df.shape)

df = df.drop(columns=['user_id', 'user_name', 'review_id', 'review_title'], errors='ignore')

df['reviews_list'] = df['review_content'].str.split(r'\s*\|\s*')
df['reviews_list'] = df['reviews_list'].apply(
    lambda x: [r.strip() for r in x if r.strip() and len(r.strip()) > 5] if isinstance(x, list) else []
)

df['reviews_found'] = df['reviews_list'].str.len()
print("\nAverage reviews per product:", round(df['reviews_found'].mean(), 1))
print("Total reviews we'll get:", df['reviews_found'].sum())

df = df.drop(columns=['review_content'])

reviews_df = df.explode('reviews_list').reset_index(drop=True)

reviews_df = reviews_df.rename(columns={'reviews_list': 'review_content'})

reviews_df = reviews_df.dropna(subset=['review_content'])
reviews_df = reviews_df[reviews_df['review_content'].str.len() > 15]  # drop tiny noise
reviews_df = reviews_df.drop(columns=['reviews_found'], errors='ignore')
reviews_df = reviews_df.reset_index(drop=True)

print("\nFINAL SHAPE (one pristine row per review):", reviews_df.shape)
display(reviews_df[['product_id', 'product_name', 'category', 'rating', 'review_content', 'rating_count']].head(15))

print("\nRating distribution (product rating repeated per review):")
print(reviews_df['rating'].value_counts().sort_index())

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

reviews_df['cleaned_review'] = reviews_df['review_content'].apply(clean_text)

print("Sample original vs cleaned:")
display(reviews_df[['review_content', 'cleaned_review']].head(10))

reviews_df = reviews_df[reviews_df['cleaned_review'].str.len() > 0]
print("\nShape after cleaning:", reviews_df.shape)

In [ ]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

reviews_df['review_length'] = reviews_df['cleaned_review'].str.split().str.len()

analyzer = SentimentIntensityAnalyzer()
reviews_df['vader_score'] = reviews_df['review_content'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

reviews_df['price_drop_pct'] = (reviews_df['actual_price'] - reviews_df['discounted_price']) / reviews_df['actual_price'] * 100

reviews_df['sentiment_label'] = (reviews_df['rating'] >= 3).astype(int)

print("New features sample:")
display(reviews_df[['review_length', 'vader_score', 'price_drop_pct', 'sentiment_label']].describe())

In [ ]:
def create_multi_sentiment(rating):
    if rating >= 4.0:
        return 'Positive'      # class 2
    elif rating >= 3.0:
        return 'Neutral'       # class 1
    else:
        return 'Negative'      # class 0

reviews_df['sentiment_multi'] = reviews_df['rating'].apply(create_multi_sentiment)

print("Binary sentiment distribution:")
print(reviews_df['sentiment_label'].value_counts(normalize=True).round(3))

print("\nMulti-class sentiment distribution:")
print(reviews_df['sentiment_multi'].value_counts())
print(reviews_df['sentiment_multi'].value_counts(normalize=True).round(3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

plt.figure(figsize=(10, 5))
sns.countplot(x='rating', data=reviews_df, palette='viridis')
plt.title('Distribution of Product Ratings')
plt.show()

reviews_df['main_category'] = reviews_df['category'].str.split('|').str[0]
plt.figure(figsize=(12, 6))
reviews_df['main_category'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 Main Categories')
plt.xticks(rotation=45)
plt.show()

top_reviewed = reviews_df['product_name'].value_counts().head(10)
least_reviewed = reviews_df['product_name'].value_counts().tail(10)

print("Most reviewed products:")
print(top_reviewed)
print("\nLeast reviewed products:")
print(least_reviewed)

plt.figure(figsize=(10, 5))
sns.boxplot(x='rating', y='review_length', data=reviews_df)
plt.title('Review Length by Rating')
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(x='rating', y='vader_score', data=reviews_df)
plt.title('VADER Sentiment Score by Rating')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, mean_squared_error, r2_score
import numpy as np

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf = vectorizer.fit_transform(reviews_df['cleaned_review'])

numeric_features = reviews_df[['review_length', 'vader_score', 'price_drop_pct']].values

from scipy.sparse import hstack
X = hstack([X_tfidf, numeric_features])

y_sentiment = reviews_df['sentiment_label']
y_rating = reviews_df['rating']

X_train, X_test, y_sent_train, y_sent_test = train_test_split(X, y_sentiment, test_size=0.2, random_state=42, stratify=y_sentiment)
_, _, y_reg_train, y_reg_test = train_test_split(X, y_rating, test_size=0.2, random_state=42)

print("TF-IDF shape:", X_tfidf.shape)
print("Full feature shape:", X.shape)
print("Positive % in data:", y_sentiment.mean().round(3))

In [ ]:
def create_multi_int(rating):
    if rating >= 4.0:
        return 2
    elif rating >= 3.0:
        return 1
    else:
        return 0

reviews_df['sentiment_multi'] = reviews_df['rating'].apply(create_multi_int)

print("Multi-class distribution:")
print(reviews_df['sentiment_multi'].value_counts().sort_index())

In [ ]:
from sklearn.metrics import classification_report

y_multi = reviews_df['sentiment_multi']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

multi_models = {
    'Logistic Regression (Multi)': LogisticRegression(max_iter=1000, multi_class='multinomial', class_weight='balanced'),
    'Random Forest (Multi)': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
}

multi_results = {}

for name, model in multi_models.items():
    model.fit(X_train_m, y_train_m)
    preds = model.predict(X_test_m)
    report = classification_report(y_test_m, preds, output_dict=True)
    multi_results[name] = {
        'Accuracy': report['accuracy'],
        'Macro F1': report['macro avg']['f1-score'],
        'Weighted F1': report['weighted avg']['f1-score']
    }

multi_df = pd.DataFrame(multi_results).T.round(3)
display(multi_df)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_sent_train)
    preds = model.predict(X_test)
    report = classification_report(y_sent_test, preds, output_dict=True)
    results[name] = {
        'Accuracy': report['accuracy'],
        'Pos Precision': report['1']['precision'],
        'Pos Recall': report['1']['recall'],
        'Pos F1': report['1']['f1-score']
    }
    print(f"\n{name} done")

import pandas as pd
results_df = pd.DataFrame(results).T.round(3)
display(results_df.sort_values('Accuracy', ascending=False))

In [ ]:
reg_models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(alpha=0.1),
    'Ridge': Ridge(alpha=1.0)
}

reg_results = {}

for name, model in reg_models.items():
    model.fit(X_train, y_reg_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_reg_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_reg_test, preds)
    reg_results[name] = {'MSE': round(mse, 3), 'RMSE': round(rmse, 3), 'R²': round(r2, 3)}
    print(f"\n{name} done")

reg_df = pd.DataFrame(reg_results).T
display(reg_df.sort_values('RMSE'))

In [ ]:
### Deep Learning: Multi-Class Sentiment with DistilBERT

!pip install -q transformers datasets evaluate accelerate

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

hf_dataset = Dataset.from_pandas(reviews_df[['cleaned_review', 'sentiment_multi']].rename(columns={'cleaned_review': 'text', 'sentiment_multi': 'label'}))

train_test = hf_dataset.train_test_split(test_size=0.2)
tokenized_datasets = train_test.map(tokenize_function, batched=True)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    }

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics
)

trainer.train()

print(trainer.evaluate())

In [ ]:
trainer.save_model("./distilbert_sentiment")

from transformers import pipeline
sentiment_pipeline = pipeline("text-classification", model="./distilbert_sentiment", tokenizer=tokenizer, return_all_scores=True)

from sklearn.linear_model import Ridge
from scipy.sparse import hstack
import numpy as np

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf = vectorizer.fit_transform(reviews_df['cleaned_review'])
numeric = reviews_df[['review_length', 'vader_score', 'price_drop_pct']].values
X_full = hstack([X_tfidf, numeric])

ridge = Ridge(alpha=1.0)
ridge.fit(X_full, reviews_df['rating'])

print("Models ready for prime time")

In [ ]:
from collections import defaultdict

product_reviews = defaultdict(list)
product_ratings = defaultdict(list)

for _, row in reviews_df.iterrows():
    product_reviews[row['product_name']].append(row['review_content'])
    product_ratings[row['product_name']].append(row['rating'])

product_avg_rating = {p: sum(r)/len(r) if r else 0 for p, r in product_ratings.items()}

print("Pre-grouped", len(product_reviews), "products")

In [ ]:
!pip install -q gradio sumy bitsandbytes accelerate vaderSentiment

import gradio as gr
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import gc
from collections import defaultdict
import pandas as pd

torch.cuda.empty_cache()
gc.collect()

product_reviews = defaultdict(list)
product_ratings = defaultdict(list)

for _, row in reviews_df.iterrows():
    name = row['product_name']
    product_reviews[name].append(row['review_content'])
    product_ratings[name].append(row['rating'])

product_avg_rating = {p: sum(r)/len(r) if r else 0 for p, r in product_ratings.items()}
products = sorted(product_reviews.keys())

vader = SentimentIntensityAnalyzer()

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct", token="")
qwen_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", token="", quantization_config=quant_config, device_map="auto")

qwen_pipe = pipeline("text-generation", model=qwen_model, tokenizer=qwen_tokenizer, max_new_tokens=256)

with gr.Blocks(theme=gr.themes.Soft(), css="""
    .wrap td, .wrap th {white-space: pre-wrap !important; word-break: break-word !important; max-width: 500px !important;}
    .dataframe-container {height: 600px !important; overflow-y: auto !important;}
""") as interface:
    gr.Markdown("# ReviewLens: Amazon Insights Hub")
    gr.Markdown("Product → sentiment, ratings, dual-mode summaries")

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            product_dropdown = gr.Dropdown(choices=products, label="Select Product", value=products[0] if products else None)
            summary_radio = gr.Radio(["Extractive (TextRank)", "Abstractive (Qwen2.5-7B)"], label="Summary Mode")
            submit_btn = gr.Button("Analyze", variant="primary")

        with gr.Column(scale=3):
            reviews_out = gr.Dataframe(label="Reviews & Ratings", wrap=True, elem_classes="wrap")
            with gr.Row():
                sentiment_out = gr.Textbox(label="Sentiment Breakdown", lines=5)
                rating_out = gr.Textbox(label="Rating Insights", lines=5)
            summary_out = gr.Textbox(label="Product Summary", lines=12, elem_classes="wrap")

    def run(product, mode):
        if product not in product_reviews:
            return pd.DataFrame(), "No product", "—", "—"

        reviews = product_reviews[product]
        ratings = product_ratings[product]
        avg = product_avg_rating[product]

        df = pd.DataFrame({'Review': reviews[:40], 'Rating': ratings[:40]})  # cap for UI sanity

        sentiments = ["Positive" if vader.polarity_scores(r)['compound'] > 0.05 else "Negative" if vader.polarity_scores(r)['compound'] < -0.05 else "Neutral" for r in reviews]
        sentiment_text = pd.Series(sentiments).value_counts().to_string() or "Perfectly balanced"

        rating_text = f"Avg rating: {avg:.2f} ⭐ ({len(reviews)} reviews)"

        full_text = " ".join(reviews)
        if mode == "Abstractive (Qwen2.5-7B)":
            prompt = f"Bullet pros/cons/insights:\n{full_text[:3000]}"
            summary = qwen_pipe(prompt)[0]['generated_text'].replace(prompt, "").strip() or "Qwen is inspired..."
        else:
            parser = PlaintextParser.from_string(full_text or "Solid", Tokenizer("english"))
            summarizer = TextRankSummarizer()
            sentences = summarizer(parser.document, 8)
            summary = "\n".join(str(s) for s in sentences) or "Consensus: good buy."

        return df, sentiment_text, rating_text, summary

    submit_btn.click(fn=run, inputs=[product_dropdown, summary_radio], outputs=[reviews_out, sentiment_out, rating_out, summary_out])

interface.launch(share=True)